In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

def obter_diretorio_input() -> Path:
    if "INPUT_DIR" in os.environ and os.path.isdir(os.environ["INPUT_DIR"]):
        return Path(os.environ["INPUT_DIR"])
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    return Path.cwd()

def obter_diretorio_output() -> Path:
    if "OUTPUT_DIR" in os.environ:
        caminho = Path(os.environ["OUTPUT_DIR"])
        caminho.mkdir(parents=True, exist_ok=True)
        return caminho
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]:
        if (p / "workflows").is_dir():
            return p / "workflows"
        if (p / "hydranomic" / "workflows").is_dir():
            return p / "hydranomic" / "workflows"
    caminho = Path.cwd() / "outputs"
    caminho.mkdir(parents=True, exist_ok=True)
    return caminho

def carregar_json(nome_arquivo: str, padrao: Any = None) -> Any:
    dir_in = obter_diretorio_input()
    candidatos = [
        dir_in / nome_arquivo,
        Path.cwd() / "workflows" / nome_arquivo,
        Path.cwd() / "hydranomic" / "workflows" / nome_arquivo,
        Path.cwd() / nome_arquivo
    ]
    for c in candidatos:
        if c.exists() and c.stat().st_size > 0:
            with open(c, "r", encoding="utf-8") as f:
                return json.load(f)
    if padrao is not None:
        return padrao
    raise FileNotFoundError(f"Arquivo {nome_arquivo} nao encontrado em {dir_in}")

def salvar_json(nome_arquivo: str, dados: Any, indent: int = 2) -> Path:
    dir_out = obter_diretorio_output()
    caminho = dir_out / nome_arquivo
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=indent)
    return caminho

def carregar_contexto() -> Dict[str, Any]:
    try:
        return carregar_json("context.json", padrao={})
    except Exception:
        return {}


In [ ]:
ESTADO = "tudo"
TARIFA_AGUA = 59.6
ARQUIVO_HISTORICO = "conta.json"
N_ESTIMATORS = 250
ANO_GRAFICO = None


In [ ]:
def gerenciar_consumo_diario_e_contas(contexto: Optional[Dict[str, Any]] = None) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    if contexto is None:
        contexto = carregar_contexto()
    if "body" in contexto and isinstance(contexto["body"], dict):
        for k, v in contexto["body"].items():
            if k not in contexto:
                contexto[k] = v
    hoje = datetime.now()
    mes_atual_id = f"{hoje.year}-{hoje.month:02d}"
    data_hoje_str = hoje.strftime("%Y-%m-%d")
    consumo_mes_padrao = {
        "mes_referencia": mes_atual_id,
        "unidade_consumidora": 14079,
        "consumo_acumulado_mes_m3": 0.0,
        "dias": []
    }
    try:
        consumo_mes = carregar_json("consumo_mes.json", padrao=consumo_mes_padrao)
    except Exception:
        consumo_mes = consumo_mes_padrao
    try:
        dados_contas = carregar_json("conta.json")
    except Exception:
        try:
            dados_contas = carregar_json("conta_povoada.json")
        except Exception:
            dados_contas = []

    mes_salvo = str(consumo_mes.get("mes_referencia", mes_atual_id))
    if mes_salvo != mes_atual_id:
        acumulado_anterior = float(consumo_mes.get("consumo_acumulado_mes_m3", 0.0))
        if acumulado_anterior == 0.0 and consumo_mes.get("dias"):
            acumulado_anterior = sum(float(d.get("consumo_m3", 0.0)) for d in consumo_mes.get("dias", []))
        if acumulado_anterior > 0:
            encontrado_ant = False
            for idx, r in enumerate(dados_contas):
                if str(r.get("id")) == mes_salvo:
                    dados_contas[idx]["consumo_m3"] = round(acumulado_anterior, 2)
                    if "valor_total" not in dados_contas[idx]:
                        dados_contas[idx]["valor_total"] = round(acumulado_anterior * TARIFA_AGUA, 2)
                    encontrado_ant = True
                    break
            if not encontrado_ant:
                dados_contas.append({
                    "id": mes_salvo,
                    "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
                    "categoria": "COMERCIAL",
                    "consumo_m3": round(acumulado_anterior, 2),
                    "valor_total": round(acumulado_anterior * TARIFA_AGUA, 2)
                })
        consumo_mes = {
            "mes_referencia": mes_atual_id,
            "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
            "consumo_acumulado_mes_m3": 0.0,
            "dias": []
        }

    dias = consumo_mes.get("dias", [])
    consumo_dia = contexto.get("consumo_dia") or contexto.get("novo_consumo_dia") or contexto.get("consumo_hoje")
    if consumo_dia is None:
        pulsos_raw = None
        if "data" in contexto and isinstance(contexto["data"], dict):
            pulsos_raw = contexto["data"].get("Pulsos") or contexto["data"].get("pulsos")
        if pulsos_raw is None:
            pulsos_raw = contexto.get("Pulsos") or contexto.get("pulsos")
        if pulsos_raw is not None:
            consumo_dia = round(float(pulsos_raw) * 0.001, 3)

    data_registro = contexto.get("data")
    if not data_registro and "time" in contexto and isinstance(contexto["time"], str):
        data_registro = contexto["time"].split("T")[0]
    if not data_registro or not isinstance(data_registro, str):
        data_registro = data_hoje_str

    try:
        dia_num = int(contexto.get("dia", data_registro.split("-")[-1]))
    except Exception:
        dia_num = hoje.day

    if consumo_dia is not None:
        try:
            consumo_float = float(consumo_dia)
            encontrado_dia = False
            for idx, d in enumerate(dias):
                if d.get("data") == data_registro or d.get("dia") == dia_num:
                    dias[idx]["consumo_m3"] = consumo_float
                    dias[idx]["data"] = data_registro
                    dias[idx]["dia"] = dia_num
                    encontrado_dia = True
                    break
            if not encontrado_dia:
                dias.append({"dia": dia_num, "data": data_registro, "consumo_m3": consumo_float})
        except Exception:
            pass

    dias = sorted(dias, key=lambda x: int(x.get("dia", 0)))
    consumo_mes["dias"] = dias
    total_acumulado = round(sum(float(d.get("consumo_m3", 0.0)) for d in dias), 2)
    consumo_mes["consumo_acumulado_mes_m3"] = total_acumulado
    salvar_json("consumo_mes.json", consumo_mes)

    nova_conta = contexto.get("nova_conta") or contexto.get("novo_registro")
    if nova_conta and isinstance(nova_conta, dict):
        id_novo = str(nova_conta.get("id", ""))
        encontrado = False
        for idx, r in enumerate(dados_contas):
            if str(r.get("id")) == id_novo:
                dados_contas[idx].update(nova_conta)
                encontrado = True
                break
        if not encontrado:
            dados_contas.append(nova_conta)
    elif total_acumulado > 0:
        encontrado_mes = False
        for idx, r in enumerate(dados_contas):
            if str(r.get("id")) == mes_atual_id:
                dados_contas[idx]["consumo_m3"] = total_acumulado
                if "valor_total" not in dados_contas[idx] or dados_contas[idx].get("categoria") == "COMERCIAL":
                    dados_contas[idx]["valor_total"] = round(total_acumulado * TARIFA_AGUA, 2)
                encontrado_mes = True
                break
        if not encontrado_mes:
            dados_contas.append({
                "id": mes_atual_id,
                "unidade_consumidora": consumo_mes.get("unidade_consumidora", 14079),
                "categoria": "COMERCIAL",
                "consumo_m3": total_acumulado,
                "valor_total": round(total_acumulado * TARIFA_AGUA, 2)
            })

    dados_contas = sorted(dados_contas, key=lambda x: str(x.get("id", "")))
    salvar_json("conta.json", dados_contas)
    return consumo_mes, dados_contas


In [ ]:
def processar_dados_grafico(dados: Optional[List[Dict[str, Any]]] = None, ano_referencia: Optional[int] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    if dados is None:
        _, dados = gerenciar_consumo_diario_e_contas(contexto)
    if ano_referencia is None:
        if "ano_referencia" in contexto:
            ano_referencia = int(contexto["ano_referencia"])
        elif ANO_GRAFICO is not None:
            ano_referencia = int(ANO_GRAFICO)
        elif dados:
            anos_encontrados = []
            for r in dados:
                partes = str(r.get("id", "")).split("-")
                if len(partes) == 2 and partes[0].isdigit():
                    anos_encontrados.append(int(partes[0]))
            ano_referencia = max(anos_encontrados) if anos_encontrados else datetime.now().year
        else:
            ano_referencia = datetime.now().year
    consumo_mensal: Dict[int, Optional[float]] = {}
    mes_atual: Optional[int] = None
    for registro in dados:
        registro_id = str(registro.get("id", ""))
        separar_partes = registro_id.split("-")
        if len(separar_partes) != 2:
            continue
        try:
            ano_registro = int(separar_partes[0])
            mes_registro = int(separar_partes[1])
        except ValueError:
            continue
        if ano_registro == ano_referencia and 1 <= mes_registro <= 12:
            consumo_mensal[mes_registro] = registro.get("consumo_m3")
            mes_atual = mes_registro
    meses_tags = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
    resultado = {
        "ano_referencia": ano_referencia,
        "mes_atual": mes_atual,
        "unidade_medida": "m³",
        "meses": [
            {
                "mes": mes,
                "rotulo": meses_tags[mes - 1],
                "consumo_m3": consumo_mensal.get(mes)
            }
            for mes in range(1, 13)
        ]
    }
    if salvar:
        salvar_json("grafico_anual.json", resultado)
    return resultado


In [ ]:
def executar_pipeline_ml(dados: Optional[List[Dict[str, Any]]] = None, tarifa: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    if tarifa is None:
        tarifa = float(contexto.get("tarifa", TARIFA_AGUA))
    if dados is None:
        _, dados = gerenciar_consumo_diario_e_contas(contexto)
    if not dados:
        try:
            dados = carregar_json("conta.json", padrao=[])
        except Exception:
            dados = []

    hoje = datetime.now()
    data_treinamento = hoje.isoformat()
    data_formatada = hoje.strftime("%d/%m/%Y %H:%M:%S")

    df = pd.DataFrame(dados) if dados else pd.DataFrame()
    if not df.empty and "id" in df.columns and "consumo_m3" in df.columns:
        df = df.dropna(subset="id", "consumo_m3").copy()
        df = df.sort_values(by=["id"]).reset_index(drop=True)
        partes = df["id"].astype(str).str.split("-")
        df["ano"] = partes.str[0].astype(int)
        df["mes"] = partes.str[1].astype(int)
        if "unidade_consumidora" not in df.columns:
            df["unidade_consumidora"] = 14079

    try:
        if df.empty or len(df) < 2:
            raise ValueError("Historico insuficiente para modelo preditivo.")

        mes_atual_id = f"{hoje.year}-{hoje.month:02d}"
        df_fechados = df[df["id"] < mes_atual_id].copy()
        if len(df_fechados) < 2:
            df_fechados = df.copy()

        ano_inicial = int(df_fechados["ano"].min())
        mes_inicial = int(df_fechados.loc[df_fechados["ano"] == ano_inicial, "mes"].min())
        df_fechados["tempo_iniciofinal"] = (df_fechados["ano"] - ano_inicial) * 12 + (df_fechados["mes"] - mes_inicial)
        df_fechados["memoria_consumo"] = df_fechados.groupby("unidade_consumidora")["consumo_m3"].shift(1)
        df_treino = df_fechados.dropna(subset=["memoria_consumo"]).copy()

        features_entrada = ["mes", "tempo_iniciofinal", "memoria_consumo"]
        variavel_saida = "consumo_m3"
        modelo = RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=45)
        modelo.fit(df_treino[features_entrada], df_treino[variavel_saida])

        ultima_linha = df.iloc[-1]
        ultimo_ano = int(ultima_linha["ano"])
        ultimo_mes = int(ultima_linha["mes"])
        ultimo_consumo = float(ultima_linha["consumo_m3"])
        ultimo_passo = int((ultimo_ano - ano_inicial) * 12 + (ultimo_mes - mes_inicial))

        if ultimo_mes == 12:
            proximo_mes = 1
            proximo_ano = ultimo_ano + 1
        else:
            proximo_mes = ultimo_mes + 1
            proximo_ano = ultimo_ano
        proximo_passo = ultimo_passo + 1

        dados_previsao = pd.DataFrame([[proximo_mes, proximo_passo, ultimo_consumo]], columns=features_entrada)
        previsao_m3 = float(modelo.predict(dados_previsao)[0])
        valor_conta = round(previsao_m3 * tarifa, 2)
        resultado = {
            "ano": proximo_ano,
            "mes": proximo_mes,
            "unidade_consumidora": int(ultima_linha.get("unidade_consumidora", 14079)),
            "ultimo_consumo_registrado_m3": ultimo_consumo,
            "previsao_m3": round(previsao_m3, 2),
            "tarifa_m3": tarifa,
            "valor_estimado_conta": valor_conta,
            "modelo": "RandomForestRegressor",
            "data_treinamento": data_treinamento,
            "data_treinamento_formatada": data_formatada
        }
    except Exception as e:
        media_consumo = float(df["consumo_m3"].mean()) if not df.empty else 300.0
        ultimo_val = float(df.iloc[-1]["consumo_m3"]) if not df.empty else media_consumo
        uc = int(df.iloc[-1]["unidade_consumidora"]) if not df.empty and "unidade_consumidora" in df.columns else 14079
        mes_prox = (hoje.month % 12) + 1
        ano_prox = hoje.year if hoje.month < 12 else hoje.year + 1
        previsao_fallback = round((media_consumo + ultimo_val) / 2.0, 2)
        resultado = {
            "ano": ano_prox,
            "mes": mes_prox,
            "unidade_consumidora": uc,
            "ultimo_consumo_registrado_m3": round(ultimo_val, 2),
            "previsao_m3": previsao_fallback,
            "tarifa_m3": tarifa,
            "valor_estimado_conta": round(previsao_fallback * tarifa, 2),
            "modelo": "Fallback_MediaMovel",
            "aviso": str(e),
            "data_treinamento": data_treinamento,
            "data_treinamento_formatada": data_formatada
        }

    if salvar:
        salvar_json("previsao_ml.json", resultado)
    return resultado


In [ ]:
def avaliar_progresso_meta(consumo_atual_m3: Optional[float] = None, salvar: bool = True) -> Dict[str, Any]:
    contexto = carregar_contexto()
    config = carregar_json("config.json", padrao={"meta_ativa": None, "solicitacao_exclusao": None})
    nova_meta = contexto.get("nova_meta")
    if nova_meta and isinstance(nova_meta, dict):
        config["meta_ativa"] = nova_meta
        config["solicitacao_exclusao"] = None
    elif contexto.get("excluir_meta") is True:
        config["meta_ativa"] = None
        config["solicitacao_exclusao"] = None
    meta = config.get("meta_ativa")
    if consumo_atual_m3 is None:
        try:
            consumo_mes = carregar_json("consumo_mes.json")
            consumo_atual_m3 = float(consumo_mes.get("consumo_acumulado_mes_m3", 0.0))
        except Exception:
            try:
                dados = carregar_json("conta.json")
                consumo_atual_m3 = float(dados[-1].get("consumo_m3", 0.0))
            except Exception:
                consumo_atual_m3 = 0.0
    if not meta:
        relatorio = {
            "meta_ativa": False,
            "mensagem": "Nenhuma meta ativa configurada.",
            "consumo_atual_m3": consumo_atual_m3
        }
    else:
        limite = float(meta.get("valor", 1.0))
        porcentagem = round((consumo_atual_m3 / limite) * 100, 2)
        saldo_restante = round(limite - consumo_atual_m3, 2)
        if porcentagem >= 100:
            situacao = "ESTOURADA"
        elif porcentagem >= 80:
            situacao = "ALERTA_80"
        else:
            situacao = "NORMAL"
        relatorio = {
            "meta_ativa": True,
            "meta_nome": meta.get("nome", "Meta de Consumo"),
            "limite_m3": limite,
            "consumo_atual_m3": consumo_atual_m3,
            "percentual_atingido": porcentagem,
            "saldo_restante_m3": saldo_restante,
            "situacao": situacao
        }
    if salvar:
        salvar_json("progresso_meta.json", relatorio)
        salvar_json("config.json", config)
    return relatorio


In [ ]:
def executar_controlador(estado_solicitado: Any = None) -> Dict[str, Any]:
    contexto = carregar_contexto()
    estado = estado_solicitado or contexto.get("estado") or contexto.get("acao") or ESTADO
    consumo_mes, dados_contas = gerenciar_consumo_diario_e_contas(contexto)
    resultado_grafico = processar_dados_grafico(dados=dados_contas, salvar=True)
    resultado_ml = executar_pipeline_ml(dados=dados_contas, salvar=True)
    consumo_mes_atual = consumo_mes.get("consumo_acumulado_mes_m3", 0.0)
    resultado_meta = avaliar_progresso_meta(consumo_atual_m3=float(consumo_mes_atual), salvar=True)
    sumario = {
        "status": "sucesso",
        "mes_atual": consumo_mes.get("mes_referencia"),
        "consumo_mes_acumulado_m3": consumo_mes.get("consumo_acumulado_mes_m3"),
        "previsao_proximo_mes_m3": resultado_ml.get("previsao_m3"),
        "fatura_estimada_reais": resultado_ml.get("valor_estimado_conta"),
        "data_treinamento_ml": resultado_ml.get("data_treinamento"),
        "ano_grafico": resultado_grafico.get("ano_referencia"),
        "meta_status": resultado_meta.get("situacao") if resultado_meta else "sem_meta_ativa"
    }
    salvar_json("sumario_execucao.json", sumario)
    resultado_execucao = {
        "estado_executado": estado,
        "timestamp": datetime.now().isoformat(),
        "outputs_gerados": [
            "conta.json",
            "consumo_mes.json",
            "grafico_anual.json",
            "previsao_ml.json",
            "progresso_meta.json",
            "config.json",
            "sumario_execucao.json"
        ],
        "ml": resultado_ml,
        "grafico": resultado_grafico,
        "metas": resultado_meta,
        "sumario": sumario
    }
    return resultado_execucao


In [ ]:
resultado = executar_controlador()
dir_saida = obter_diretorio_output()
arquivos = list(dir_saida.glob("*.json"))
print(f"Execucao concluida. Arquivos no OUTPUT_DIR ({dir_saida}):")
for a in arquivos:
    print(f" - {a.name} ({a.stat().st_size} bytes)")
